# SPR 2026 - Classificação Hierárquica (Cascata) com BERTimbau v2

**Estratégia:** Classificação em cascata (benigno vs alterado)

## ⚠️ ALERTA DOS ORGANIZADORES

> "Two of the classes have a relatively low prevalence in the test set, which means that performance on those classes can have a disproportionate impact on the final score."

**Classes raras prováveis:** 5 (Altamente Sugestivo) e 6 (Malignidade Comprovada)

**Melhorias nesta versão:**
- ✅ Stratified K-Fold CV (5 folds) para validação robusta
- ✅ Análise F1 per-class para monitorar classes raras
- ✅ OOF predictions para estimativa confiável
- ✅ Threshold tuning focado em classes minoritárias

## Arquitetura Hierárquica

| Modelo | Tarefa | Classes |
|--------|--------|----------|
| Modelo 1 | Benigno vs Alterado (binário) | 0=Benigno, 1=Alterado |
| Modelo 2 | Classificar Benignos | 1, 2, 3 (Negativo, Benigno, Prob. Benigno) |
| Modelo 3 | Classificar Alterados | 0, 4, 5, 6 (Incompleto, Suspeito, Alt. Sugestivo, Malignidade) |

## Mapeamento BI-RADS

**BENIGNO (1, 2, 3):**
- 1: Negativo (nenhuma alteração)
- 2: Benigno (achado benigno)
- 3: Provavelmente Benigno

**ALTERADO (0, 4, 5, 6):**
- 0: Incompleto (necessita avaliação adicional)
- 4: Suspeito
- 5: Altamente Sugestivo de Malignidade ⚠️ **RARA**
- 6: Malignidade Comprovada ⚠️ **RARA**

---
## 📥 MODELO - Baixar com: `models/download_bertimbau_large.ipynb`

**Kaggle Models:** Add Input → Models → `bertimbau-ptbr-complete`

---
**CONFIGURAÇÃO KAGGLE:**
1. Settings → Internet → **OFF**
2. Settings → Accelerator → **GPU T4 x2**
---

In [ ]:
# ===== SPR 2026 - CASCATA BERTIMBAU v2 (K-FOLD + RARE CLASS FOCUS) =====

print("[1/12] Configurando ambiente...")
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ========== CONFIGURAÇÕES GLOBAIS ==========
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

# K-Fold para validação robusta
N_FOLDS = 5
USE_KFOLD = True  # False = single split (mais rápido mas menos robusto)

# Mapeamento de classes
CLASSES_BENIGNO = [1, 2, 3]  # Negativo, Benigno, Prob. Benigno
CLASSES_ALTERADO = [0, 4, 5, 6]  # Incompleto, Suspeito, Alt. Sugestivo, Malignidade
CLASSES_RARAS = [5, 6]  # Classes com baixa prevalência (alerta dos organizadores)

BIRADS_LABELS = {
    0: 'Incompleto', 1: 'Negativo', 2: 'Benigno', 3: 'Provavelmente Benigno',
    4: 'Suspeito', 5: 'Altamente Sugestivo', 6: 'Malignidade Comprovada'
}

# =============================================

DATA_DIR = '/kaggle/input/competitions/spr-2026-mammography-report-classification'

# Verificar dataset
if not os.path.exists(DATA_DIR):
    DATA_DIR = '/kaggle/input/spr-2026-mammography-report-classification'
    if not os.path.exists(DATA_DIR):
        print("\n" + "="*60)
        print("ERRO: Dataset não encontrado!")
        print("="*60)
        print("\nAdicione o dataset:")
        print("Add Input → Competition → spr-2026-mammography-report-classification")
        raise FileNotFoundError(f"Dataset não encontrado")
print(f"Dataset: {DATA_DIR}")

# Auto-detectar modelo BERTimbau
def find_model_path():
    base = '/kaggle/input'
    def has_config(path):
        return os.path.isdir(path) and os.path.exists(os.path.join(path, 'config.json'))
    def search_dir(directory, depth=0, max_depth=10):
        if depth > max_depth:
            return None
        try:
            for item in os.listdir(directory):
                path = os.path.join(directory, item)
                if os.path.isdir(path):
                    if has_config(path):
                        return path
                    result = search_dir(path, depth + 1, max_depth)
                    if result:
                        return result
        except:
            pass
        return None
    return search_dir(base)

MODEL_PATH = find_model_path()
if MODEL_PATH is None:
    raise FileNotFoundError("Adicione o modelo BERTimbau ao notebook!")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model: {MODEL_PATH}')
print(f'K-Fold: {N_FOLDS} folds' if USE_KFOLD else 'Single Split (10%)')
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Focal Loss
print("[2/10] Definindo Focal Loss...")

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print(f'Focal Loss: gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA}')

In [ ]:
# Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# Carregar dados
print("[3/12] Carregando dados...")
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

# ========== ANÁLISE DE DISTRIBUIÇÃO DAS CLASSES ==========
print("\n" + "="*60)
print("⚠️  ANÁLISE DE DISTRIBUIÇÃO (ALERTA DOS ORGANIZADORES)")
print("="*60)

class_counts = train_df['target'].value_counts().sort_index()
total = len(train_df)

print("\nDistribuição no treino:")
for c, count in class_counts.items():
    pct = count / total * 100
    rara = "⚠️ RARA" if c in CLASSES_RARAS else ""
    bar = "█" * int(pct / 2)
    print(f"  Classe {c} ({BIRADS_LABELS[c]:25}): {count:5} ({pct:5.1f}%) {bar} {rara}")

print(f"\n📊 Classes raras identificadas: {CLASSES_RARAS}")
print("   → Performance nestas classes tem impacto desproporcional no F1-Macro!")
print("="*60)

# Criar label binária: Benigno(0) vs Alterado(1)
def map_to_binary(target):
    if target in CLASSES_BENIGNO:
        return 0  # Benigno
    else:
        return 1  # Alterado

train_df['target_binary'] = train_df['target'].apply(map_to_binary)

print("\nDistribuição Binária:")
print(f"  Benigno (0):  {(train_df['target_binary'] == 0).sum()}")
print(f"  Alterado (1): {(train_df['target_binary'] == 1).sum()}")

# Separar subsets
train_benigno = train_df[train_df['target'].isin(CLASSES_BENIGNO)].copy()
train_alterado = train_df[train_df['target'].isin(CLASSES_ALTERADO)].copy()

print(f"\nSubset Benigno: {len(train_benigno)} amostras")
print(f"Subset Alterado: {len(train_alterado)} amostras (contém classes raras 5 e 6)")

In [ ]:
# Função de treinamento com análise per-class
def train_model_fold(train_texts, train_labels, val_texts, val_labels, num_classes, model_name="Modelo", 
                     class_names=None, return_probs=False):
    """Treina um fold e retorna modelo + métricas detalhadas."""
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds = TextDataset(val_texts, val_labels, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=num_classes, local_files_only=True
    )
    model.to(device)
    
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    best_f1 = 0
    best_model_state = None
    best_val_probs = None
    best_val_preds = None
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        # Validação com probabilidades
        model.eval()
        val_preds, val_true, val_probs = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = F.softmax(outputs.logits, dim=1)
                preds = outputs.logits.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(batch['labels'].numpy())
                val_probs.extend(probs.cpu().numpy())
        
        val_f1 = f1_score(val_true, val_preds, average='macro')
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict().copy()
            best_val_probs = np.array(val_probs)
            best_val_preds = np.array(val_preds)
    
    # Carregar melhor modelo
    model.load_state_dict(best_model_state)
    
    # Análise per-class F1
    per_class_f1 = f1_score(val_true, best_val_preds, average=None)
    
    result = {
        'model': model,
        'tokenizer': tokenizer,
        'best_f1': best_f1,
        'per_class_f1': per_class_f1,
        'val_true': np.array(val_true),
        'val_preds': best_val_preds,
        'val_probs': best_val_probs
    }
    
    return result


def train_model_kfold(texts, labels, num_classes, model_name="Modelo", class_names=None):
    """Treina com K-Fold CV e retorna métricas agregadas + modelo final."""
    
    print(f"\n{'='*60}")
    print(f"Treinando {model_name}")
    print(f"Classes: {num_classes}, Amostras: {len(texts)}, Folds: {N_FOLDS}")
    print(f"{'='*60}")
    
    if class_names is None:
        class_names = [str(i) for i in range(num_classes)]
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    fold_f1s = []
    all_per_class_f1 = []
    oof_preds = np.zeros(len(texts))
    oof_probs = np.zeros((len(texts), num_classes))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
        print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
        
        train_texts_fold = texts[train_idx]
        train_labels_fold = labels[train_idx]
        val_texts_fold = texts[val_idx]
        val_labels_fold = labels[val_idx]
        
        result = train_model_fold(
            train_texts_fold, train_labels_fold,
            val_texts_fold, val_labels_fold,
            num_classes, model_name, class_names
        )
        
        fold_f1s.append(result['best_f1'])
        all_per_class_f1.append(result['per_class_f1'])
        oof_preds[val_idx] = result['val_preds']
        oof_probs[val_idx] = result['val_probs']
        
        print(f"Fold {fold+1} F1-Macro: {result['best_f1']:.4f}")
    
    # Métricas OOF (Out-of-Fold) - estimativa mais confiável
    oof_f1 = f1_score(labels, oof_preds, average='macro')
    oof_per_class = f1_score(labels, oof_preds, average=None)
    
    print(f"\n{'='*60}")
    print(f"{model_name} - RESULTADOS K-FOLD")
    print(f"{'='*60}")
    print(f"Média F1 (folds):  {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
    print(f"OOF F1-Macro:      {oof_f1:.4f}  ← USAR PARA COMPARAÇÃO")
    
    print(f"\nF1 por classe (OOF):")
    for i, (name, f1) in enumerate(zip(class_names, oof_per_class)):
        rara = "⚠️" if f1 < 0.5 else ""
        print(f"  {name:25}: {f1:.4f} {rara}")
    
    # Treinar modelo final em todos os dados
    print(f"\nTreinando modelo final em 100% dos dados...")
    final_result = train_model_fold(
        texts, labels,
        texts[:100], labels[:100],  # Val dummy para estrutura
        num_classes, model_name
    )
    
    return {
        'model': final_result['model'],
        'tokenizer': final_result['tokenizer'],
        'oof_f1': oof_f1,
        'oof_per_class': oof_per_class,
        'oof_preds': oof_preds,
        'oof_probs': oof_probs,
        'fold_f1s': fold_f1s
    }


def train_model_simple(texts, labels, num_classes, model_name="Modelo", class_names=None):
    """Treina com single split (mais rápido, menos robusto)."""
    
    print(f"\n{'='*60}")
    print(f"Treinando {model_name}")
    print(f"Classes: {num_classes}, Amostras: {len(texts)}")
    print(f"{'='*60}")
    
    if class_names is None:
        class_names = [str(i) for i in range(num_classes)]
    
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.1, random_state=SEED, stratify=labels
    )
    
    result = train_model_fold(
        train_texts, train_labels,
        val_texts, val_labels,
        num_classes, model_name, class_names
    )
    
    print(f"\nVal F1-Macro: {result['best_f1']:.4f}")
    print(f"\nF1 por classe:")
    for i, (name, f1) in enumerate(zip(class_names, result['per_class_f1'])):
        rara = "⚠️" if f1 < 0.5 else ""
        print(f"  {name:25}: {f1:.4f} {rara}")
    
    return {
        'model': result['model'],
        'tokenizer': result['tokenizer'],
        'oof_f1': result['best_f1'],
        'oof_per_class': result['per_class_f1'],
        'fold_f1s': [result['best_f1']]
    }

In [ ]:
# ========== MODELO 1: Benigno vs Alterado (BINÁRIO) ==========
print("[4/12] Treinando Modelo 1: Benigno vs Alterado...")

texts_1 = train_df['report'].values
labels_1 = train_df['target_binary'].values
class_names_1 = ['Benigno (1,2,3)', 'Alterado (0,4,5,6)']

if USE_KFOLD:
    result_binary = train_model_kfold(
        texts_1, labels_1, num_classes=2,
        model_name="Modelo 1 (Benigno vs Alterado)",
        class_names=class_names_1
    )
else:
    result_binary = train_model_simple(
        texts_1, labels_1, num_classes=2,
        model_name="Modelo 1 (Benigno vs Alterado)",
        class_names=class_names_1
    )

model_binary = result_binary['model']
tokenizer = result_binary['tokenizer']
f1_binary = result_binary['oof_f1']

print(f"\n✅ Modelo 1 treinado - OOF F1: {f1_binary:.4f}")

In [ ]:
# ========== MODELO 2: Classificar Benignos (1, 2, 3) ==========
print("[5/12] Treinando Modelo 2: Classificar Benignos...")

# Mapear labels para 0, 1, 2 (original: 1, 2, 3)
label_map_benigno = {1: 0, 2: 1, 3: 2}
label_map_benigno_inv = {0: 1, 1: 2, 2: 3}

texts_2 = train_benigno['report'].values
labels_2 = train_benigno['target'].map(label_map_benigno).values
class_names_2 = ['1-Negativo', '2-Benigno', '3-Prob. Benigno']

if USE_KFOLD:
    result_benigno = train_model_kfold(
        texts_2, labels_2, num_classes=3,
        model_name="Modelo 2 (Benignos: 1,2,3)",
        class_names=class_names_2
    )
else:
    result_benigno = train_model_simple(
        texts_2, labels_2, num_classes=3,
        model_name="Modelo 2 (Benignos: 1,2,3)",
        class_names=class_names_2
    )

model_benigno = result_benigno['model']
f1_benigno = result_benigno['oof_f1']

print(f"\n✅ Modelo 2 treinado - OOF F1: {f1_benigno:.4f}")

In [ ]:
# ========== MODELO 3: Classificar Alterados (0, 4, 5, 6) - CONTÉM CLASSES RARAS ==========
print("[6/12] Treinando Modelo 3: Classificar Alterados (com classes raras 5 e 6)...")

# Mapear labels para 0, 1, 2, 3 (original: 0, 4, 5, 6)
label_map_alterado = {0: 0, 4: 1, 5: 2, 6: 3}
label_map_alterado_inv = {0: 0, 1: 4, 2: 5, 3: 6}

texts_3 = train_alterado['report'].values
labels_3 = train_alterado['target'].map(label_map_alterado).values
class_names_3 = ['0-Incompleto', '4-Suspeito', '5-Alt.Sugestivo ⚠️', '6-Malignidade ⚠️']

print("\n⚠️  ATENÇÃO: Modelo 3 contém as classes raras (5 e 6)")
print("   → Monitorar F1 destas classes de perto!")

# Distribuição no subset alterado
print("\nDistribuição no subset alterado:")
for orig, mapped in label_map_alterado.items():
    count = (train_alterado['target'] == orig).sum()
    pct = count / len(train_alterado) * 100
    rara = "⚠️ RARA" if orig in CLASSES_RARAS else ""
    print(f"  Classe {orig} ({BIRADS_LABELS[orig]:25}): {count:5} ({pct:5.1f}%) {rara}")

if USE_KFOLD:
    result_alterado = train_model_kfold(
        texts_3, labels_3, num_classes=4,
        model_name="Modelo 3 (Alterados: 0,4,5,6)",
        class_names=class_names_3
    )
else:
    result_alterado = train_model_simple(
        texts_3, labels_3, num_classes=4,
        model_name="Modelo 3 (Alterados: 0,4,5,6)",
        class_names=class_names_3
    )

model_alterado = result_alterado['model']
f1_alterado = result_alterado['oof_f1']
f1_per_class_alterado = result_alterado['oof_per_class']

print(f"\n✅ Modelo 3 treinado - OOF F1: {f1_alterado:.4f}")

# Alerta específico para classes raras
if len(f1_per_class_alterado) >= 4:
    f1_classe5 = f1_per_class_alterado[2]  # mapeado para índice 2
    f1_classe6 = f1_per_class_alterado[3]  # mapeado para índice 3
    print(f"\n📊 Performance classes raras:")
    print(f"   Classe 5 (Alt. Sugestivo): F1 = {f1_classe5:.4f}")
    print(f"   Classe 6 (Malignidade):    F1 = {f1_classe6:.4f}")
    if f1_classe5 < 0.5 or f1_classe6 < 0.5:
        print("   ⚠️ ALERTA: Classes raras com F1 < 0.5 - pode haver shake-up no private LB!")

In [ ]:
# Resumo do treinamento
print("[7/12] Resumo do treinamento...")
print("="*60)
print("CASCATA - RESUMO DOS MODELOS (OOF = Out-of-Fold)")
print("="*60)
print(f"\nModelo 1 (Benigno vs Alterado): OOF F1 = {f1_binary:.4f}")
print(f"Modelo 2 (Benignos 1,2,3):       OOF F1 = {f1_benigno:.4f}")
print(f"Modelo 3 (Alterados 0,4,5,6):    OOF F1 = {f1_alterado:.4f}")

print("\n" + "-"*60)
print("⚠️  AVALIAÇÃO PARA PRIVATE LEADERBOARD")
print("-"*60)
print("\nUsar OOF F1 para selecionar submissões (não confiar só no public LB)!")
print("Classes raras (5, 6) têm impacto desproporcional no F1-Macro.")

if USE_KFOLD:
    print(f"\n✅ Validação K-Fold ({N_FOLDS} folds) executada - estimativa confiável")
else:
    print(f"\n⚠️ Single split usado - executar com USE_KFOLD=True para estimativa mais robusta")

print("="*60)

In [ ]:
# ========== THRESHOLD TUNING PARA CLASSES RARAS ==========
print("[8/12] Definindo thresholds para classes raras...")

# Thresholds ajustados para aumentar recall das classes raras
# Modelo 1: Benigno vs Alterado
THRESHOLD_BINARY = 0.45  # Mais sensível para Alterado (contém classes raras)

# Modelo 3: Thresholds por classe (0, 4, 5, 6 mapeados para 0, 1, 2, 3)
THRESHOLDS_ALTERADO = {
    0: 0.50,  # Incompleto
    1: 0.50,  # Suspeito
    2: 0.30,  # Alt. Sugestivo (RARA) - mais sensível
    3: 0.25,  # Malignidade (RARA) - muito mais sensível
}

print("\n📊 Thresholds configurados:")
print(f"   Binário (Alterado): {THRESHOLD_BINARY} (< 0.5 = mais sensível para alterados)")
print("\n   Modelo 3 (Alterados):")
for idx, t in THRESHOLDS_ALTERADO.items():
    nome = ['0-Incompleto', '4-Suspeito', '5-Alt.Sugestivo ⚠️', '6-Malignidade ⚠️'][idx]
    print(f"      {nome}: {t}")

def apply_threshold_binary(probs, threshold=0.5):
    """Aplica threshold para classificação binária."""
    # probs[:, 1] = probabilidade de ser Alterado
    return (probs[:, 1] >= threshold).astype(int)

def apply_thresholds_multiclass(probs, thresholds):
    """Aplica thresholds por classe, priorizando classes raras."""
    preds = []
    for i in range(len(probs)):
        adj_probs = probs[i].copy()
        # Ajustar probabilidades baseado nos thresholds
        for c, t in thresholds.items():
            if c < len(adj_probs):
                # Boost para classes com threshold menor
                adj_probs[c] *= (0.5 / t)
        preds.append(np.argmax(adj_probs))
    return np.array(preds)

print("\n✅ Funções de threshold tuning definidas")

In [ ]:
# ========== INFERÊNCIA EM CASCATA ==========
print("[9/12] Inferência em cascata no conjunto de teste...")

# Criar dataset de teste
test_ds = TextDataset(test_df['report'].values, None, tokenizer, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# Passo 1: Classificação binária (Benigno vs Alterado) COM THRESHOLD
print("\nPasso 1: Classificação Binária (com threshold tuning)")
model_binary.eval()
binary_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Modelo 1 - Binário'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model_binary(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        binary_probs.extend(probs.cpu().numpy())

binary_probs = np.array(binary_probs)

# Aplicar threshold (menor que 0.5 = mais sensível para classe 1 = Alterado)
binary_preds = apply_threshold_binary(binary_probs, THRESHOLD_BINARY)

print(f"Threshold: {THRESHOLD_BINARY}")
print(f"Benigno (0): {(binary_preds == 0).sum()}")
print(f"Alterado (1): {(binary_preds == 1).sum()}")

In [ ]:
# Passo 2: Classificar benignos (1, 2, 3)
print("\nPasso 2: Classificar Benignos")
indices_benigno = np.where(binary_preds == 0)[0]
print(f"Amostras a classificar: {len(indices_benigno)}")

benigno_preds = np.zeros(len(test_df), dtype=int)

if len(indices_benigno) > 0:
    model_benigno.eval()
    
    # Criar subset para benignos
    texts_benigno = test_df['report'].values[indices_benigno]
    ds_benigno = TextDataset(texts_benigno, None, tokenizer, MAX_LEN)
    loader_benigno = DataLoader(ds_benigno, batch_size=BATCH_SIZE)
    
    preds_benigno_temp = []
    with torch.no_grad():
        for batch in tqdm(loader_benigno, desc='Modelo 2 - Benignos'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_benigno(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            preds_benigno_temp.extend(preds.cpu().numpy())
    
    # Mapear de volta para classes originais (0,1,2 -> 1,2,3)
    preds_benigno_mapped = [label_map_benigno_inv[p] for p in preds_benigno_temp]
    benigno_preds[indices_benigno] = preds_benigno_mapped
    
    print(f"Distribuição benignos:")
    for c in [1, 2, 3]:
        count = sum(1 for p in preds_benigno_mapped if p == c)
        print(f"  Classe {c}: {count}")

In [ ]:
# Passo 3: Classificar alterados (0, 4, 5, 6) COM THRESHOLD TUNING para CLASSES RARAS
print("\nPasso 3: Classificar Alterados (com threshold tuning para classes raras)")
indices_alterado = np.where(binary_preds == 1)[0]
print(f"Amostras a classificar: {len(indices_alterado)}")

alterado_preds = np.zeros(len(test_df), dtype=int)
alterado_probs_all = np.zeros((len(test_df), 4))

if len(indices_alterado) > 0:
    model_alterado.eval()
    
    # Criar subset para alterados
    texts_alterado = test_df['report'].values[indices_alterado]
    ds_alterado = TextDataset(texts_alterado, None, tokenizer, MAX_LEN)
    loader_alterado = DataLoader(ds_alterado, batch_size=BATCH_SIZE)
    
    probs_alterado_temp = []
    with torch.no_grad():
        for batch in tqdm(loader_alterado, desc='Modelo 3 - Alterados'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_alterado(input_ids=input_ids, attention_mask=attention_mask)
            probs = F.softmax(outputs.logits, dim=1)
            probs_alterado_temp.extend(probs.cpu().numpy())
    
    probs_alterado_temp = np.array(probs_alterado_temp)
    
    # Aplicar threshold tuning (boost para classes raras 5 e 6)
    preds_alterado_temp = apply_thresholds_multiclass(probs_alterado_temp, THRESHOLDS_ALTERADO)
    
    # Mapear de volta para classes originais (0,1,2,3 -> 0,4,5,6)
    preds_alterado_mapped = [label_map_alterado_inv[p] for p in preds_alterado_temp]
    alterado_preds[indices_alterado] = preds_alterado_mapped
    alterado_probs_all[indices_alterado] = probs_alterado_temp
    
    print(f"\nDistribuição alterados (com threshold tuning):")
    for c in [0, 4, 5, 6]:
        count = sum(1 for p in preds_alterado_mapped if p == c)
        rara = "⚠️" if c in CLASSES_RARAS else ""
        print(f"  Classe {c} ({BIRADS_LABELS[c]:25}): {count:5} {rara}")

In [ ]:
# Combinar predições finais
print("[11/12] Combinando predições finais...")

final_predictions = np.zeros(len(test_df), dtype=int)

# Atribuir predições de benignos
final_predictions[indices_benigno] = benigno_preds[indices_benigno]

# Atribuir predições de alterados
final_predictions[indices_alterado] = alterado_preds[indices_alterado]

print("\n" + "="*60)
print("DISTRIBUIÇÃO FINAL DAS PREDIÇÕES")
print("="*60)
for c in range(7):
    count = (final_predictions == c).sum()
    rara = "⚠️ RARA" if c in CLASSES_RARAS else ""
    print(f"  Classe {c} ({BIRADS_LABELS[c]:25}): {count:5} {rara}")

# Sanity check para classes raras
count_5 = (final_predictions == 5).sum()
count_6 = (final_predictions == 6).sum()
print(f"\n📊 Classes raras preditas:")
print(f"   Classe 5 (Alt. Sugestivo): {count_5}")
print(f"   Classe 6 (Malignidade):    {count_6}")
if count_5 == 0 or count_6 == 0:
    print("   ⚠️ ALERTA: Classes raras sem predições - pode impactar F1-Macro!")

In [ ]:
# Gerar submissão
print("[12/12] Gerando submissão...")

sample_path = f'{DATA_DIR}/sample_submission.csv'
if os.path.exists(sample_path):
    sample_sub = pd.read_csv(sample_path)
    SUB_ID = sample_sub.columns[0]
    SUB_LABEL = sample_sub.columns[1]
else:
    SUB_ID = 'ID'
    SUB_LABEL = 'target'

submission = pd.DataFrame({SUB_ID: test_df['ID'], SUB_LABEL: final_predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print("\n" + "="*70)
print("CASCATA BERTIMBAU v2 - CONCLUÍDO!")
print("="*70)

print("\n📊 MÉTRICAS OOF (Out-of-Fold) - USAR PARA SELEÇÃO DE SUBMISSÕES:")
print("-"*70)
print(f"Modelo 1 (Benigno vs Alterado): OOF F1 = {f1_binary:.4f}")
print(f"Modelo 2 (Benignos 1,2,3):       OOF F1 = {f1_benigno:.4f}")
print(f"Modelo 3 (Alterados 0,4,5,6):    OOF F1 = {f1_alterado:.4f}")

print("\n📈 DISTRIBUIÇÃO DAS PREDIÇÕES:")
print("-"*70)
for c in range(7):
    count = (final_predictions == c).sum()
    rara = "⚠️" if c in CLASSES_RARAS else ""
    print(f"  Classe {c}: {count:5} {rara}")

print("\n" + "="*70)
print("⚠️  RECOMENDAÇÕES PARA PRIVATE LEADERBOARD:")
print("="*70)
print("1. Use métricas OOF (não public LB) para selecionar 2 melhores submissões")
print("2. Classes 5 e 6 são raras - impactam desproporcionalmente o F1-Macro")
print("3. Espere shake-up entre public e private leaderboard")
print(f"4. Validação: {'K-Fold' if USE_KFOLD else 'Single Split'} - ", end="")
print("confiável" if USE_KFOLD else "considere executar com USE_KFOLD=True")
print("="*70)

print("\n✅ submission.csv criado!")